In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

import datetime as dt
from pathlib import Path



In [2]:
def extract_Ap(Ap_file):

    full_data_df = pd.read_csv(filepath_or_buffer='raw_data/' + Ap_file,
                               sep=r'\s+',
                               skiprows=39)
    
    year_mm_dd = full_data_df[['#YYY', 'MM', 'DD']].copy()

    year_mm_dd = year_mm_dd.rename(columns={'#YYY': 'year', 'MM': 'month', 'DD': 'day'})

    dates = pd.to_datetime(year_mm_dd)
    
    Ap_df = pd.DataFrame({'Datetime': dates, 'Ap': full_data_df['Ap']})

    # Add thing to deal with bad Ap values

    #Ap_df.to_csv('Ap.csv', index=False)

    return Ap_df

In [3]:
Ap_file = 'Kp_ap_Ap_SN_F107_since_1932.txt'

Ap_df = extract_Ap(Ap_file=Ap_file)

print(Ap_df)

        Datetime  Ap
0     1932-01-01  15
1     1932-01-02  26
2     1932-01-03  11
3     1932-01-04   4
4     1932-01-05   3
...          ...  ..
34286 2025-11-14   3
34287 2025-11-15   8
34288 2025-11-16  16
34289 2025-11-17  10
34290 2025-11-18   4

[34291 rows x 2 columns]


### Extract Data From Reports

In [10]:
# Extract three_day_forecast
# Figure out how to store three_day_forecast in csv
# Extract geomag_forecast
# Figure out how to store geomag_forecast in csv
# Extract the_weekly
# Figure out how to store the_weekly in csv

# Look into RSGA report for Tom

# Register for SWPC space weather workshop and poster presentation

In [11]:
# See if daypre forecast is always on same line
# Maybe use exceptions to make sure ti all works
# Figure out how to take line

In [80]:
def get_daypre_lines(fill_nans=True):
    
    report_name = 'daypre'
    dir_path = Path(f'raw_data/{report_name}')

    last_ind = 8
    raw_dt_strings = [p.name[:last_ind] for p in dir_path.iterdir() if p.is_file()]
    raw_file_strings = [p for p in dir_path.iterdir() if p.is_file()]
    unsorted_dt_list = [dt.date(year=int(p[0:4]), month=int(p[4:6]), day=int(p[6:8])) for p in raw_dt_strings]
    
    # Initialize lists to store daily Ap forecast
    ap_lists = [[] for i in range(3)]

    # Extract daily Ap forecast
    for file in raw_file_strings:

        with open(file, 'r', encoding='utf-8') as f:

            for line in f:
                
                if 'A_Planetary' in line:
                    
                    parts = line.split()

                    ap_lists[0].append(parts[1])
                    ap_lists[1].append(parts[2])
                    ap_lists[2].append(parts[3])
                    continue
    
    ap_dict = {'date': unsorted_dt_list}
    for i in range(len(ap_lists)):
        ap_dict[f'{i+1}dAp'] = ap_lists[i]

    # Store in dataframe       
    df_daypre = pd.DataFrame(ap_dict)
    
    # Sort dataframe so that dates are in order
    df_daypre.sort_values('date', inplace=True)
    df_daypre.set_index(pd.Index([i for i in range(0, len(df_daypre['date']))]), inplace=True)

    if fill_nans is True:
        
        df_daypre = df_daypre.set_index('date')
        full_range = pd.date_range(start=df_daypre.index.min(), end=df_daypre.index.max(), freq='D')
        df_daypre = df_daypre.reindex(full_range)
        df_daypre = df_daypre.reset_index().rename(columns={'index': 'date'})

    df_daypre.to_csv('processed_data/daypre_3dayAp.csv')
    
    return df_daypre

In [81]:
get_daypre_lines()

,date,1dAp,2dAp,3dAp
0,2013-12-22,5,5,10
1,2013-12-23,5,10,8
2,2013-12-24,10,8,5
3,2013-12-25,10,8,5
4,2013-12-26,7,5,5
...,...,...,...,...
4434,2026-02-11,5,5,12
4435,2026-02-12,5,12,20
4436,2026-02-13,12,25,20
4437,2026-02-14,25,20,12


In [ ]:
def extract_geomag_forecast_kp_from_file(file, kp_lists):

    with open(file, 'r', encoding='utf-8') as f:
        
            lines = [line for line in f]

            for i in range(len(lines)):

                if '00-03UT' in lines[i]:

                    lines = lines[i:i+8]

                    break
            
            for i in range(len(lines)):
                
                parts = lines[i].split()

                kp_lists[i].append(parts[1])
                kp_lists[i+8].append(parts[2])
                kp_lists[i+16].append(parts[3])

    return

In [ ]:
def get_geomag_forecast(fill_nans=True):
    
    report_name = 'geomag_forecast'
    dir_path = Path(f'raw_data/{report_name}')

    last_ind = 8
    raw_dt_strings = [p.name[:last_ind] for p in dir_path.iterdir() if p.is_file()]
    raw_file_strings = [p for p in dir_path.iterdir() if p.is_file()]
    unsorted_dt_list = [dt.date(year=int(p[0:4]), month=int(p[4:6]), day=int(p[6:8])) for p in raw_dt_strings]

    kp_lists = [[] for i in range(24)]
    
    # Extract Kp forecast
    for file in raw_file_strings:

        extract_geomag_forecast_kp_from_file(file=file,
                             kp_lists=kp_lists)
        
    kp_dict = {'date': unsorted_dt_list}
    for i in range(len(kp_lists)):
        kp_dict[f'{3*(i+1)}hrKp'] = kp_lists[i]

    # Store in dataframe               
    df_geomag_forecast_Kp = pd.DataFrame(kp_dict)
    
    # Sort dataframe so that dates are in order
    df_geomag_forecast_Kp.sort_values('date', inplace=True)
    df_geomag_forecast_Kp.set_index(pd.Index([i for i in range(0, len(df_geomag_forecast_Kp['date']))]), inplace=True)

    # Fill missing forecast dates with Nan
    if fill_nans is True:
        
        df_geomag_forecast_Kp = df_geomag_forecast_Kp.set_index('date')
        full_range = pd.date_range(start=df_geomag_forecast_Kp.index.min(), end=df_geomag_forecast_Kp.index.max(), freq='D')
        df_geomag_forecast_Kp = df_geomag_forecast_Kp.reindex(full_range)
        df_geomag_forecast_Kp = df_geomag_forecast_Kp.reset_index().rename(columns={'index': 'date'})

    df_geomag_forecast_Kp.to_csv('processed_data/geomag_forecast_3dayKp.csv')
    
    return df_geomag_forecast_Kp

In [77]:
get_geomag_forecast()

,date,3hrKp,6hrKp,9hrKp,12hrKp,15hrKp,18hrKp,21hrKp,24hrKp,27hrKp,...,45hrKp,48hrKp,51hrKp,54hrKp,57hrKp,60hrKp,63hrKp,66hrKp,69hrKp,72hrKp
0,2022-11-13,2,1,1,1,1,1,2,2,2,...,3,3,3,3,3,3,3,2,2,2
1,2022-11-14,2.00,2.00,1.33,1.33,2.00,3.00,3.00,3.00,2.67,...,2.00,2.33,1.67,1.33,1.33,1.33,1.33,1.33,1.67,1.67
2,2022-11-15,1.67,2.67,2.00,1.67,1.33,1.33,2.00,2.67,1.67,...,1.67,1.67,1.67,1.67,1.33,1.67,2.33,2.33,2.67,2.67
3,2022-11-16,1.67,1.33,1.33,1.33,1.33,1.33,1.67,1.67,1.33,...,2.67,2.33,2.00,2.00,2.00,2.33,2.33,3.33,4.00,4.33
4,2022-11-17,1.33,1.67,3.00,2.67,2.33,2.33,2.67,2.67,2.67,...,4.00,4.67,4.67,4.33,5.33,3.33,2.67,3.00,3.67,4.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1186,2026-02-11,1.67,2.00,1.67,1.33,0.67,1.00,1.67,1.67,1.67,...,1.67,1.67,3.00,2.67,2.00,2.67,2.00,2.67,2.67,3.00
1187,2026-02-12,1.67,1.33,1.33,1.33,1.33,1.33,1.67,1.67,3.00,...,2.67,3.00,3.67,2.67,2.67,2.33,2.33,2.33,4.67,5.00
1188,2026-02-13,3.00,2.67,2.00,2.67,2.00,2.67,2.67,3.67,2.67,...,3.33,4.67,4.67,4.33,4.00,3.00,2.67,1.67,2.67,3.33
1189,2026-02-14,2.67,4.33,2.67,3.33,3.67,4.67,3.67,4.67,4.67,...,2.67,3.33,2.67,3.67,3.00,2.67,2.67,1.67,1.67,2.67


In [3]:
import src.extract_forecasts_utils as ecu

display(ecu.get_geomag_forecast()[1])

,date,1dAp,2dAp,3dAp
0,2022-11-13,5.0,10.0,10.0
1,2022-11-14,10.0,10.0,5.0
2,2022-11-15,8.0,5.0,8.0
3,2022-11-16,5.0,10.0,15.0
4,2022-11-17,10.0,18.0,28.0
...,...,...,...,...
1186,2026-02-11,5.0,5.0,12.0
1187,2026-02-12,5.0,12.0,20.0
1188,2026-02-13,12.0,25.0,20.0
1189,2026-02-14,25.0,20.0,12.0
